In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import os
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv(r"C:\Users\vasth\Downloads\gwl_tel_6_hourly_meghalaya_ml_2021_2025.csv")

time_col = 'Data Acquisition Time'
target_col = 'Groundwater Level Telemetry 6 Hourly (meter)'

df[time_col] = pd.to_datetime(df[time_col], format='%d-%m-%Y %H:%M', errors='coerce')

df = df.dropna(subset=[time_col, target_col])

df = df[(df[target_col] >= -50) & (df[target_col] < 150)]

df = df.sort_values(by=['Station', time_col])

# Feature Engineering: Temporal features
df['Year'] = df[time_col].dt.year
df['Month'] = df[time_col].dt.month
df['Day'] = df[time_col].dt.day
df['Hour'] = df[time_col].dt.hour
df['DayOfYear'] = df[time_col].dt.dayofyear

df['GWL_lag1'] = df.groupby('Station')[target_col].shift(1)
df['GWL_lag2'] = df.groupby('Station')[target_col].shift(2)
df['GWL_lag4'] = df.groupby('Station')[target_col].shift(4)

df['GWL_diff_1'] = df['GWL_lag1'] - df['GWL_lag2']

df['GWL_roll_mean_24h'] = df.groupby('Station')['GWL_lag1'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df['GWL_roll_mean_48h'] = df.groupby('Station')['GWL_lag1'].transform(lambda x: x.rolling(window=8, min_periods=1).mean())
df['GWL_roll_std_24h'] = df.groupby('Station')['GWL_lag1'].transform(lambda x: x.rolling(window=4, min_periods=2).std())

df = df.dropna(subset=['GWL_lag1', 'GWL_lag2', 'GWL_lag4', 'GWL_diff_1', 'GWL_roll_mean_24h', 'GWL_roll_std_24h'])

features = [
    'Year', 'Month', 'Day', 'Hour', 'DayOfYear', 
    'GWL_lag1', 'GWL_lag2', 'GWL_lag4', 
    'GWL_diff_1', 'GWL_roll_mean_24h', 'GWL_roll_mean_48h', 'GWL_roll_std_24h'
]

if len(df['Station'].unique()) > 1:
    df['Station_Code'] = df['Station'].astype('category').cat.codes
    features.append('Station_Code')

X = df[features]
y = df[target_col]

In [3]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

xgb_model = XGBRegressor(
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=6, 
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0, 
    random_state=42, 
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [4]:
predictions = xgb_model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:.4f} meters")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} meters")

Mean Absolute Error (MAE): 0.9303 meters
Root Mean Squared Error (RMSE): 3.7687 meters


In [5]:
import folium
from IPython.display import display

stations_df = df.dropna(subset=['Latitude', 'Longitude'])[['Station', 'Latitude', 'Longitude']].drop_duplicates(subset=['Station'])

if len(stations_df) > 0:
    m = folium.Map(location=[stations_df['Latitude'].mean(), stations_df['Longitude'].mean()], tiles='OpenStreetMap')

    for _, row in stations_df.iterrows():
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=7, color='black', weight=1, fill=True,
            fill_color='#1f77b4', fill_opacity=0.85,
            tooltip=str(row['Station']),
            popup=folium.Popup(str(row['Station']), max_width=200)
        ).add_to(m)

    m.fit_bounds(stations_df[['Latitude', 'Longitude']].agg(['min', 'max']).values.tolist())
    m.get_root().html.add_child(folium.Element('<h3 align="center">Water Well Locations (Meghalaya Dataset)</h3>'))

    map_path = r"D:\GW Prediction\water_wells_map.html"
    m.save(map_path)
    print(f"Map saved to: {map_path}")
    display(m)
else:
    print("Warning: Latitude/Longitude columns may be missing or empty.")

Map saved to: D:\GW Prediction\water_wells_map.html
